In [15]:
"""
Data processing module for product review analysis.
Handles data loading, preprocessing, and train/test splitting.
"""

import pandas as pd
import numpy as np
import re
from typing import Tuple, Dict
from sklearn.feature_extraction.text import TfidfVectorizer


def load_review_data(file_path: str) -> pd.DataFrame:
    """
    Load product review data from CSV file.

    Args:
        file_path: Path to CSV file containing review data

    Returns:
        DataFrame with review data

    Raises:
        FileNotFoundError: If file doesn't exist
        ValueError: If required columns are missing
    """
    REQUIRED_COLUMNS = ["review_text", "sentiment"]

    data = pd.read_csv(file_path)

    if not all(col in data.columns for col in REQUIRED_COLUMNS):
        raise ValueError("Missing required columns")

    data.dropna(subset=REQUIRED_COLUMNS, inplace=True)
    return data


def preprocess_text(text: str) -> str:
    """
    Preprocess review text for analysis.

    Args:
        text: Raw review text

    Returns:
        Preprocessed text (lowercase, cleaned)
    """
    lower_text = text.lower()
    lower_text = re.sub(r"[^a-z0-9\s]", "", lower_text)
    lower_text = re.sub(r"\s+", " ", lower_text).strip()
    return lower_text


def split_data(
    df: pd.DataFrame,
    test_size: float = 0.15,
    val_size: float = 0.15,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split data into train, validation, and test sets.

    Args:
        df: DataFrame containing review data
        test_size: Fraction of data for test set
        val_size: Fraction of data for validation set
        random_state: Random seed for reproducibility

    Returns:
        Tuple of (train_df, val_df, test_df)

    Note:
        Ensure no data leakage between splits!
    """
    df_shuffled = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    train_end = int((1 - test_size - val_size) * len(df_shuffled))
    val_end = int((1 - test_size) * len(df_shuffled))

    train_df = df_shuffled[:train_end]
    val_df = df_shuffled[train_end:val_end]
    test_df = df_shuffled[val_end:]

    return train_df, val_df, test_df


def create_features(
    train_texts: pd.Series,
    val_texts: pd.Series,
    test_texts: pd.Series,
    max_features: int = 1000,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, TfidfVectorizer]:
    """
    Create TF-IDF feature representations from text data.

    Args:
        train_texts: Training text data
        val_texts: Validation text data
        test_texts: Test text data
        max_features: Maximum number of features to extract

    Returns:
        Tuple of (train_features, val_features, test_features, vectorizer)

    Important:
        - Fit vectorizer ONLY on training data
        - Transform validation and test data using the fitted vectorizer
        - This prevents data leakage
    """
    vectorizer = TfidfVectorizer(max_features=max_features)
    train_features = vectorizer.fit_transform(train_texts).toarray()
    val_features = vectorizer.transform(val_texts).toarray()
    test_features = vectorizer.transform(test_texts).toarray()
    return train_features, val_features, test_features, vectorizer


In [16]:
"""
Feature extraction module for product reviews.
Extracts product features and attributes using pattern matching.
"""

import re
from typing import Dict, List, Set
from collections import Counter


class FeatureExtractor:
    """
    Extracts product features mentioned in reviews using regex patterns.
    """

    def __init__(self):
        """Initialize feature patterns and categories."""
        self.feature_patterns = {
            # Add patterns for different feature categories
            # Example: "battery": r"\b(battery|power|charge)\b"
            "battery": r"\b(battery|power|charge)\b",
            "screen": r"\b(screen|display|resolution)\b",
            "camera": r"\b(camera|photo|picture)\b",
            "performance": r"\b(performance|speed|lag)\b",
            "design": r"\b(design|build|quality)\b",
        }

        self.feature_categories = {
            # Map features to categories
            # Example: "battery": "performance"
            "battery": "performance",
            "screen": "design",
            "camera": "quality",
            "performance": "performance",
            "design": "design",
        }

    def extract_features(self, text: str) -> Dict[str, List[str]]:
        """
        Extract product features from review text.

        Args:
            text: Review text

        Returns:
            Dictionary mapping feature categories to lists of mentioned features
        """
        features_mentions = {category: [] for category in self.feature_categories.values()}
        words = [word.lower() for word in text.split()]
        for word in words:
            for category, pattern in self.feature_patterns.items():
                if re.search(pattern, word):
                    features_mentions[self.feature_categories[category]].append(word)
        return features_mentions

    def get_feature_mentions(self, reviews: List[str]) -> Dict[str, int]:
        """
        Count feature mentions across multiple reviews.

        Args:
            reviews: List of review texts

        Returns:
            Dictionary mapping features to mention counts
        """
        # TODO: Implement feature counting
        # 1. Extract features from all reviews
        # 2. Count occurrences
        # 3. Return sorted by frequency
        reviews_features = {}

        for review in reviews:
            features = self.extract_features(review)
            reviews_features[review] = features

        return reviews_features

    def extract_sentiment_phrases(self, text: str) -> Dict[str, List[str]]:
        """
        Extract positive and negative phrases from text.

        Args:
            text: Review text

        Returns:
            Dictionary with 'positive' and 'negative' phrase lists
        """
        # TODO: Implement sentiment phrase extraction
        # Use patterns to identify positive phrases (e.g., "love", "excellent", "great")
        # and negative phrases (e.g., "terrible", "poor", "disappointed")
        positive_patterns = [
            r"[^.!?]*\b(?:love|excellent|great|amazing|fantastic)\b[^.!?]*[.!?]?",
            r"[^.!?]*\b(?:best|wonderful|superb|positive|happy)\b[^.!?]*[.!?]?",
        ]

        negative_patterns = [
            r"[^.!?]*\b(?:terrible|poor|disappointed|bad|awful)\b[^.!?]*[.!?]?",
            r"[^.!?]*\b(?:worst|horrible|negative|sad|angry)\b[^.!?]*[.!?]?",
        ]

        positive_mentions = []
        negative_mentions = []

        for pattern in positive_patterns:
            for match in re.findall(pattern, text, flags=re.IGNORECASE):
                print(match.strip())

        for pattern in negative_patterns:
            for match in re.findall(pattern, text, flags=re.IGNORECASE):
                print(match.strip())

        return {
            "positive": positive_mentions,
            "negative": negative_mentions
        }

    def summarize_features(self, reviews: List[str], top_n: int = 5) -> Dict[str, any]:
        """
        Generate feature summary from reviews.

        Args:
            reviews: List of review texts
            top_n: Number of top features to include

        Returns:
            Dictionary with feature statistics and top mentions
        """
        # 1. Extract all features
        all_features = self.get_feature_mentions(reviews)
        # 2. Calculate statistics (frequency, percentage)
        total_mentions = sum(all_features.values())
        feature_stats = {
            feature: {
                "count": count,
                "percentage": (count / total_mentions * 100) if total_mentions > 0 else 0
            }
            for feature, count in all_features.items()
        }
        # 3. Identify top features
        top_features = sorted(feature_stats.items(), key=lambda x: x[1]["count"], reverse=True)[:top_n]
        # 4. Return comprehensive summary
        return {
            "total_reviews": len(reviews),
            "feature_statistics": feature_stats,
            "top_features": top_features
        }


In [17]:
"""
Review ranking module using attention-based similarity scoring.
Ranks reviews by relevance to a given question.
"""

import numpy as np
from typing import List, Tuple, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime

class ReviewRanker:
    """
    Ranks reviews by relevance using attention-based scoring.
    Combines semantic similarity with metadata signals.
    """

    def __init__(self):
        """Initialize ranker with TF-IDF vectorizer."""
        self.vectorizer = TfidfVectorizer()
        self.review_embeddings = None

    def fit(self, reviews: List[str]) -> None:
        """
        Fit ranker on review corpus.

        Args:
            reviews: List of review texts
        """
        # 1. Initialize and fit TF-IDF vectorizer on reviews
        self.vectorizer = TfidfVectorizer()
        # 2. Compute and store review embeddings
        self.review_embeddings = self.vectorizer.fit_transform(reviews).toarray()

    def compute_similarity(self, query: str, reviews: List[str]) -> np.ndarray:
        """
        Compute cosine similarity between query and reviews.

        Args:
            query: Question or search query
            reviews: List of review texts

        Returns:
            Array of similarity scores (one per review)
        """
        # 1. Transform query to embedding using fitted vectorizer
        query_embedding = self.vectorizer.transform([query]).toarray()
        # 2. Compute cosine similarity with review embeddings
        similarity_scores = cosine_similarity(query_embedding, self.review_embeddings)
        # 3. Return similarity scores
        return similarity_scores.flatten()

    def rank_reviews(
        self,
        query: str,
        reviews: List[str],
        review_metadata: List[Dict] = None,
        top_k: int = 5
    ) -> List[Tuple[int, float, str]]:
        """
        Rank reviews by relevance to query.

        Args:
            query: Question or search query
            reviews: List of review texts
            review_metadata: Optional metadata (verified_purchase, date, rating)
            top_k: Number of top reviews to return

        Returns:
            List of tuples: (review_index, relevance_score, review_text)
            Sorted by relevance score (highest first)
        """
        # 1. Compute semantic similarity scores
        scores = self.compute_similarity(query, reviews)
        # 2. Apply metadata boosting if available:
        #    - Boost verified purchases (+0.1)
        if review_metadata and review_metadata.get("verified_purchase", False):
            scores += 0.1
        #    - Boost recent reviews (+0.05 if within 30 days)
        if review_metadata and review_metadata.get("date"):
            review_date = review_metadata["date"]
            if (datetime.now() - review_date).days <= 30:
                scores += 0.05
        #    - Boost detailed reviews (length > 100 chars, +0.05)
        if len(query) > 100:
            scores += 0.05
        # 3. Sort by final score
        sorted_indices = np.argsort(scores)[::-1][:top_k]
        # 4. Return top_k results
        return [(i, scores[i], reviews[i]) for i in sorted_indices]

    def explain_ranking(
        self,
        query: str,
        review: str,
        score: float
    ) -> Dict[str, any]:
        """
        Explain why a review was ranked highly.

        Args:
            query: Question or search query
            review: Review text
            score: Relevance score

        Returns:
            Dictionary with explanation details
        """
        # 1. Identify overlapping keywords
        overlapping_keywords = set(query.split()).intersection(set(review.split()))
        # 2. Calculate contribution of different signals
        contribution = {
            "semantic_similarity": score,
            "overlapping_keywords": list(overlapping_keywords),
            # Additional contributions can be added here
        }
        # 3. Return structured explanation
        return contribution

    def get_attention_weights(self, query: str, review: str) -> Dict[str, float]:
        """
        Compute attention weights for words in review relative to query.
        Simplified attention mechanism for interpretability.

        Args:
            query: Question text
            review: Review text

        Returns:
            Dictionary mapping review words to attention weights
        """
        # 1. Tokenize query and review
        query_tokens = query.split()
        review_tokens = review.split()
        # 2. For each review word, compute relevance to query
        relevance = {word: self.compute_similarity(word, query) for word in review_tokens}
        # 3. Normalize weights to sum to 1
        total = sum(relevance.values())
        attention_weights = {word: weight / total for word, weight in relevance.items()} if total > 0 else {}
        # 4. Return word-level attention weights
        return attention_weights


In [ ]:
"""
Neural network implementation for sentiment classification.
Binary classification: positive (1) vs negative (0) sentiment.
"""

import numpy as np
from typing import Dict, Tuple, List


class SentimentNetwork:
    """
    2-layer feedforward neural network for binary sentiment classification.
    Architecture: input -> hidden (ReLU) -> output (sigmoid)
    """

    def __init__(self, input_size: int, hidden_size: int = 64, learning_rate: float = 0.01):
        """
        Initialize network with random weights.

        Args:
            input_size: Number of input features
            hidden_size: Number of neurons in hidden layer
            learning_rate: Learning rate for gradient descent
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate

        self.w1 = np.random.randn(self.input_size, self.hidden_size) * np.sqrt(2. / self.input_size)
        self.b1 = np.zeros(self.hidden_size)
        self.w2 = np.random.randn(self.hidden_size, 1) * np.sqrt(2. / self.hidden_size)
        self.b2 = 0

    def relu(self, x: np.ndarray) -> np.ndarray:
        """ReLU activation function."""
        return np.maximum(0, x)

    def relu_derivative(self, x: np.ndarray) -> np.ndarray:
        """Derivative of ReLU function."""
        return np.where(x > 0, 1, 0)

    def sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Sigmoid activation function."""
        x = np.clip(x, -709, 709)  # Clip to avoid overflow
        return 1 / (1 + np.exp(-x))

    def forward(self, X: np.ndarray) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        """
        Forward propagation through the network.

        Args:
            X: Input features, shape (batch_size, input_size)

        Returns:
            Tuple of (predictions, cache)
            - predictions: Output probabilities, shape (batch_size, 1)
            - cache: Dictionary with intermediate values needed for backprop
        """
        # 1. Hidden layer: z1 = X @ w1 + b1, a1 = relu(z1)
        z1 = X @ self.w1 + self.b1
        a1 = self.relu(z1)

        # 2. Output layer: z2 = a1 @ w2 + b2, predictions = sigmoid(z2)
        z2 = a1 @ self.w2 + self.b2
        predictions = self.sigmoid(z2)

        # 3. Store intermediate values in cache for backprop
        cache = {
            "X": X,
            "z1": z1,
            "a1": a1,
            "z2": z2,
            "predictions": predictions
        }
        return predictions, cache

    def backward(
        self,
        X: np.ndarray,
        y: np.ndarray,
        cache: Dict[str, np.ndarray]
    ) -> Dict[str, np.ndarray]:
        """
        Backward propagation to compute gradients.

        Args:
            X: Input features, shape (batch_size, input_size)
            y: True labels, shape (batch_size, 1)
            cache: Cached values from forward pass

        Returns:
            Dictionary of gradients for all parameters
        """
        # 1. Compute output layer gradients
        dz2 = cache["predictions"] - y
        dw2 = cache["a1"].T @ dz2
        db2 = np.sum(dz2, axis=0)

        # 2. Compute hidden layer gradients using chain rule
        dz1 = dz2 @ self.w2.T * self.relu_derivative(cache["z1"])
        dw1 = cache["X"].T @ dz1
        db1 = np.sum(dz1, axis=0)

        # 3. Return gradients for w1, b1, w2, b2
        return {
            "w1": dw1,
            "b1": db1,
            "w2": dw2,
            "b2": db2
        }

    def update_weights(self, gradients: Dict[str, np.ndarray]) -> None:
        """
        Update weights using gradient descent.

        Args:
            gradients: Dictionary of gradients for all parameters
        """
        self.w1 -= self.learning_rate * gradients["w1"]
        self.b1 -= self.learning_rate * gradients["b1"]
        self.w2 -= self.learning_rate * gradients["w2"]
        self.b2 -= self.learning_rate * gradients["b2"]

    def compute_loss(self, predictions: np.ndarray, y: np.ndarray) -> float:
        """
        Compute binary cross-entropy loss.

        Args:
            predictions: Predicted probabilities, shape (batch_size, 1)
            y: True labels, shape (batch_size, 1)

        Returns:
            Average loss across batch
        """
        # Loss = -mean(y * log(pred) + (1-y) * log(1-pred))
        loss = -np.mean(y * np.log(predictions + 1e-15) + (1 - y) * np.log(1 - predictions + 1e-15))
        return loss

    def train_epoch(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray
    ) -> float:
        """
        Train for one epoch.

        Args:
            X_train: Training features
            y_train: Training labels

        Returns:
            Average loss for the epoch
        """
        # 1. Forward pass
        predictions, cache = self.forward(X_train)
        # 2. Compute loss
        loss = self.compute_loss(predictions, y_train)
        # 3. Backward pass
        gradients = self.backward(X_train, y_train, cache)
        # 4. Update weights
        self.update_weights(gradients)
        return loss

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """
        Make predictions on new data.

        Args:
            X: Input features
            threshold: Classification threshold

        Returns:
            Binary predictions (0 or 1)
        """
        # 1. Forward pass
        predictions, _ = self.forward(X)
        # 2. Apply threshold to get binary predictions
        return (predictions > threshold).astype(int)

    def evaluate(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        """
        Evaluate model performance.

        Args:
            X: Input features
            y: True labels

        Returns:
            Dictionary with accuracy, precision, recall, f1 scores
        """
        # Calculate: accuracy, precision, recall, F1 score
        preds = self.predict(X)
        accuracy = np.mean(preds == y)
        precision = np.sum((preds == 1) & (y == 1)) / np.sum(preds == 1) if np.sum(preds == 1) > 0 else 0
        recall = np.sum((preds == 1) & (y == 1)) / np.sum(y == 1) if np.sum(y == 1) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }


def train_with_early_stopping(
    network: SentimentNetwork,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 100,
    patience: int = 10,
    min_delta: float = 0.005
) -> Dict[str, List[float]]:
    """
    Train network with early stopping.

    Args:
        network: SentimentNetwork instance
        X_train: Training features
        y_train: Training labels
        X_val: Validation features
        y_val: Validation labels
        epochs: Maximum number of epochs
        patience: Number of epochs to wait for improvement
        min_delta: Minimum change in validation loss to qualify as improvement

    Returns:
        Dictionary with training history (train_loss, val_loss)
    """
    # 1. Track validation loss
    train_loss_history = []
    val_loss_history = []
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(epochs):
        # Train for one epoch
        train_loss = network.train_epoch(X_train, y_train)
        train_loss_history.append(train_loss)

        # Validate
        val_predictions, _ = network.forward(X_val)
        val_loss = network.compute_loss(val_predictions, y_val)
        val_loss_history.append(val_loss)

        # 2. Stop if validation loss doesn't improve by at least min_delta for 'patience' epochs
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

    # 3. Return training history
    return {
        "train_loss": train_loss_history,
        "val_loss": val_loss_history
    }


In [ ]:
"""
Cost optimization module for LLM usage.
Implements caching, cost tracking, and budget management.
"""

import hashlib
import json
from typing import Dict, Optional, List
from collections import defaultdict


class CostOptimizer:
    """
    Manages LLM costs through caching and tracking.
    """

    def __init__(self, budget_limit: float = 1.0):
        """
        Initialize cost optimizer.

        Args:
            budget_limit: Maximum allowed cost in dollars
        """
        self.budget_limit = budget_limit
        self.total_cost = 0.0
        self.cache = {}
        self.cache_hits = 0
        self.cache_misses = 0

        # Token costs per 1K tokens (example rates)
        self.cost_per_1k_tokens = {
            "gpt-4.1-mini": {"prompt": 0.001, "completion": 0.0015},
            "gpt-3.5-turbo": {"prompt": 0.0015, "completion": 0.002},
            "gpt-4": {"prompt": 0.03, "completion": 0.06},
            "gpt-4-turbo": {"prompt": 0.01, "completion": 0.03},
        }

    def create_cache_key(self, question: str, context: List[str]) -> str:
        """
        Create stable cache key from question and context.

        Args:
            question: Question text
            context: List of context strings (reviews)

        Returns:
            Stable hash string for caching

        Important:
            Use hashlib (MD5/SHA256), NOT Python's hash() function!
            Python's hash() is not stable across sessions.
        """
        # 1. Combine question and context into canonical form
        combined = json.dumps({"question": question, "context": context}, sort_keys=True)
        # 2. Use hashlib.md5() or hashlib.sha256() for stable hashing
        hash_object = hashlib.md5(combined.encode()).hexdigest()
        # 3. Return hex digest as cache key
        return hash_object

    def get_cached_response(self, cache_key: str) -> Optional[Dict]:
        """
        Retrieve cached response if available.

        Args:
            cache_key: Cache key

        Returns:
            Cached response dictionary or None if not cached
        """
        # 1. Check if key exists in cache
        if self.cache.get(cache_key) is not None:
            self.cache_hits += 1
            return self.cache[cache_key]
        self.cache_misses += 1
        return None

    def cache_response(self, cache_key: str, response: Dict) -> None:
        """
        Store response in cache.

        Args:
            cache_key: Cache key
            response: Response dictionary to cache
        """
        self.cache[cache_key] = response

    def calculate_cost(
        self,
        model: str,
        prompt_tokens: int,
        completion_tokens: int
    ) -> float:
        """
        Calculate cost for LLM API call.

        Args:
            model: Model name
            prompt_tokens: Number of prompt tokens
            completion_tokens: Number of completion tokens

        Returns:
            Cost in dollars
        """
        # 1. Get cost rates for model
        # 2. Calculate: (prompt_tokens / 1000) * prompt_rate +
        #               (completion_tokens / 1000) * completion_rate
        total = (prompt_tokens / 1000) * self.cost_per_1k_tokens[model]["prompt"] + \
                (completion_tokens / 1000) * self.cost_per_1k_tokens[model]["completion"]
        # 3. Return total cost
        return total

    def track_cost(
        self,
        model: str,
        prompt_tokens: int,
        completion_tokens: int
    ) -> float:
        """
        Track cost of API call and update total.

        Args:
            model: Model name
            prompt_tokens: Number of prompt tokens
            completion_tokens: Number of completion tokens

        Returns:
            Cost for this call
        """
        # 1. Calculate cost
        total = self.calculate_cost(model, prompt_tokens, completion_tokens)
        # 2. Add to total_cost
        self.total_cost += total
        # 3. Check budget limit and warn if exceeded
        if not self.check_budget():
            print("Warning: Budget exceeded!")
        # 4. Return cost
        return total

    def check_budget(self) -> bool:
        """
        Check if we're within budget.

        Returns:
            True if within budget, False otherwise
        """
        return self.total_cost <= self.budget_limit

    def get_cache_stats(self) -> Dict[str, any]:
        """
        Get cache performance statistics.

        Returns:
            Dictionary with cache metrics:
            - total_requests: Total number of requests
            - cache_hits: Number of cache hits
            - cache_misses: Number of cache misses
            - hit_rate: Cache hit rate (0-1)
        """
        # Calculate hit rate: hits / (hits + misses)
        return {
            "total_requests": self.cache_hits + self.cache_misses,
            "cache_hits": self.cache_hits,
            "cache_misses": self.cache_misses,
            "hit_rate": self.cache_hits / (self.cache_hits + self.cache_misses) if (self.cache_hits + self.cache_misses) > 0 else 0
        }

    def estimate_cost(
        self,
        model: str,
        estimated_prompt_tokens: int,
        estimated_completion_tokens: int
    ) -> Dict[str, any]:
        """
        Estimate cost before making API call.

        Args:
            model: Model name
            estimated_prompt_tokens: Estimated prompt tokens
            estimated_completion_tokens: Estimated completion tokens

        Returns:
            Dictionary with:
            - estimated_cost: Estimated cost in dollars
            - within_budget: Whether this call would exceed budget
            - remaining_budget: Remaining budget after this call
        """
        # 1. Calculate estimated cost
        estimated_cost = self.calculate_cost(model, estimated_prompt_tokens, estimated_completion_tokens)
        # 2. Check if within budget
        within_budget = (self.total_cost + estimated_cost) <= self.budget_limit
        # 3. Calculate remaining budget
        remaining_budget = self.budget_limit - (self.total_cost + estimated_cost) if not within_budget else self.budget_limit - self.total_cost
        # 4. Return estimation details
        return {
            "estimated_cost": estimated_cost,
            "within_budget": within_budget,
            "remaining_budget": remaining_budget
        }

    def reset_stats(self) -> None:
        """Reset all statistics and cache."""
        self.cache_hits = 0
        self.cache_misses = 0
        self.total_cost = 0.0

    def get_cost_summary(self) -> Dict[str, any]:
        """
        Get comprehensive cost summary.

        Returns:
            Dictionary with cost metrics and cache statistics
        """
        cache_hit_rate = self.cache_hits / (self.cache_hits + self.cache_misses) if (self.cache_hits + self.cache_misses) > 0 else 0
        return {
            "total_cost": self.total_cost,
            "budget_limit": self.budget_limit,
            "within_budget": self.check_budget(),
            "cache_hit_rate": cache_hit_rate,
            "total_requests": self.cache_hits + self.cache_misses,
        }

In [ ]:
"""
LLM client for answering questions using product reviews.
Implements proper parameter tuning, retry logic, and error handling.
"""

import time
import asyncio
from typing import List, Dict, Optional
from openai import AsyncOpenAI


class LLMClient:
    """
    Client for interacting with LLM API to answer questions.
    Implements best practices for production use.
    """

    def __init__(
        self,
        api_key: str,
        model: str = "gpt-4.1-mini",
        base_url: Optional[str] = None
    ):
        """
        Initialize LLM client.

        Args:
            api_key: OpenAI API key
            model: Model name to use
            base_url: Optional base URL for API (for compatible APIs)
        """
        self.api_key = api_key
        self.model = model
        self.base_url = base_url
        self.client = AsyncOpenAI(api_key=api_key, base_url=base_url)

    def count_tokens(self, text: str) -> int:
        """
        Estimate token count for text.

        Args:
            text: Input text

        Returns:
            Estimated token count

        Note:
            Simple estimation: ~4 characters per token
            For production, use tiktoken library
        """
        return len(text) // 4

    def build_prompt(
        self,
        question: str,
        relevant_reviews: List[str],
        max_context_tokens: int = 2000
    ) -> str:
        """
        Build prompt for question answering.

        Args:
            question: Customer question
            relevant_reviews: List of relevant review texts
            max_context_tokens: Maximum tokens for context

        Returns:
            Formatted prompt string
        """
        # 1. Create system prompt explaining task
        system_prompt = """
        You are a helpful assistant analyzing product reviews to answer customer questions.
        Based on the following reviews, provide a concise and accurate answer.
        """
        # 2. Add relevant reviews as context (respecting max_context_tokens)
        idx = 0
        context = ""

        while self.count_tokens(context) < max_context_tokens and idx < len(relevant_reviews):
            context += f"{relevant_reviews[idx]}\n"
            idx += 1

        # 3. Add the question
        prompt = f"{system_prompt}\n\nReviews:\n{context}\n\nQuestion: {question}\n\n"
        # 4. Return complete prompt
        return prompt

    def answer_question(
        self,
        question: str,
        relevant_reviews: List[str],
        temperature: float = 0.3,
        max_tokens: int = 150,
        top_p: float = 0.9
    ) -> Dict[str, any]:
        """
        Answer question using LLM with relevant reviews as context.

        Args:
            question: Customer question
            relevant_reviews: List of relevant review texts
            temperature: Sampling temperature (0.0-1.0)
                        Lower = more deterministic
            max_tokens: Maximum tokens in response
            top_p: Nucleus sampling parameter

        Returns:
            Dictionary with:
            - answer: Generated answer text
            - tokens_used: Total tokens consumed
            - model: Model used
        """
        # 1. Build prompt from question and reviews
        prompt = self.build_prompt(question, relevant_reviews)
        # 2. Call LLM API with proper parameters using asyncio.run():
        response = asyncio.run(
            self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                top_p=top_p
            )
        )
        # 3. Extract answer from response
        answer = response.choices[0].message.content
        # 4. Count tokens used (prompt + completion)
        tokens_used = self.count_tokens(prompt) + response.usage.total_tokens
        # 5. Return structured result
        return {
            "answer": answer,
            "tokens_used": tokens_used,
            "model": self.model
        }

    def answer_with_retry(
        self,
        question: str,
        relevant_reviews: List[str],
        max_retries: int = 3,
        **llm_params
    ) -> Dict[str, any]:
        """
        Answer question with exponential backoff retry logic.

        Args:
            question: Customer question
            relevant_reviews: List of relevant review texts
            max_retries: Maximum number of retry attempts
            **llm_params: Additional parameters for LLM call

        Returns:
            Answer dictionary or raises exception after max retries
        """
        # 1. Try to answer question
        try:
            response = self.answer_question(question, relevant_reviews, **llm_params)
            return response
        except Exception as e:
            last_exception = e

        # 2. If API error occurs, retry with exponential backoff
        for i in range(max_retries):
            wait_time = 2 ** i  # Exponential backoff
            time.sleep(wait_time)
            try:
                response = self.answer_question(question, relevant_reviews, **llm_params)
                return response
            except Exception as e:
                last_exception = e

        # 3. After max_retries, raise the last exception
        raise last_exception

        # 4. Return successful result
        return response

    def stream_answer(
        self,
        question: str,
        relevant_reviews: List[str],
        **llm_params
    ):
        """
        Stream answer tokens for better UX.

        Args:
            question: Customer question
            relevant_reviews: List of relevant review texts
            **llm_params: Additional parameters for LLM call

        Yields:
            Token chunks as they arrive
        """
        # 1. Build prompt
        prompt = self.build_prompt(question, relevant_reviews)

        # 2. Handle errors gracefully
        try:
            # 3. Call API with stream=True using asyncio.run()
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                stream=True,
                **llm_params
            )
            # 4. Yield chunks as they arrive
            for chunk in response:
                yield chunk
        except Exception as e:
            self.logger.error(f"Error streaming response: {e}")
            yield {"error": str(e)}

    def validate_parameters(
        self,
        temperature: float,
        max_tokens: int,
        top_p: float
    ) -> None:
        """
        Validate LLM parameters.

        Args:
            temperature: Should be in [0.0, 1.0]
            max_tokens: Should be positive and reasonable
            top_p: Should be in [0.0, 1.0]

        Raises:
            ValueError: If parameters are invalid
        """
        # Check ranges and raise descriptive errors
        if not (0.0 <= temperature <= 1.0):
            raise ValueError("Temperature must be between 0.0 and 1.0.")
        if max_tokens <= 0:
            raise ValueError("max_tokens must be positive.")
        if not (0.0 <= top_p <= 1.0):
            raise ValueError("top_p must be between 0.0 and 1.0.")